# Classificação — Pré-processamento e Baseline

**Objetivo:** a partir do entendimento construído na EDA (`Classificacao_EDA.ipynb`), preparar os dados para modelagem e estabelecer um baseline de referência.

**Dataset:** acidentes em rodovias federais (PRF, 2024). **Alvo:** `com_vitima_fatal` (binário, desbalanceado ~93%/7%).

**Regra de ouro:** o split treino/teste acontece **antes** de qualquer transformação (encoding, escala) para evitar data leakage. As funções de carregamento/preparação ficam centralizadas em `src/data.py` para serem reaproveitadas pelos próximos notebooks.

## 1. Imports

In [ ]:
import sys
sys.path.append('..')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
from sklearn.pipeline import Pipeline

from src.data import ALVO, RANDOM_STATE, TEST_SIZE, carregar_dados, construir_preprocessador, separar_x_y

sns.set_theme(style='whitegrid')

## 2. Carregar dados e separar features/alvo

In [ ]:
df = carregar_dados()
X, y = separar_x_y(df)
print('X:', X.shape, '| y positivos:', y.mean().round(4))

## 3. Split treino/teste (antes de qualquer transformação)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

print('Treino:', X_train.shape, '| positivos:', y_train.mean().round(4))
print('Teste :', X_test.shape, '| positivos:', y_test.mean().round(4))

## 4. Baseline ingênuo

Um `DummyClassifier` que sempre prevê a classe majoritária tem *accuracy* alta só por causa do desbalanceamento — é a referência mínima que qualquer modelo real precisa superar (olhando F1/ROC-AUC, não accuracy).

In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
pred_dummy = dummy.predict(X_test)
print(classification_report(y_test, pred_dummy, zero_division=0))

## 5. Baseline real: Regressão Logística

`class_weight='balanced'` compensa o desbalanceamento re-ponderando as classes na função de perda, sem precisar reamostrar os dados.

In [ ]:
pipeline_baseline = Pipeline([
    ('preprocessador', construir_preprocessador()),
    ('modelo', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)),
])
pipeline_baseline.fit(X_train, y_train)

pred = pipeline_baseline.predict(X_test)
proba = pipeline_baseline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred))
print('ROC-AUC:', round(roc_auc_score(y_test, proba), 4))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, pred, cmap='Blues')
plt.title('Matriz de confusão — baseline (LogReg)')
plt.savefig('../reports/baseline_matriz_confusao.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Salvar o baseline

In [ ]:
joblib.dump(pipeline_baseline, '../models/baseline_logreg.joblib')

## 7. Próximos passos
- Comparar múltiplos modelos com validação cruzada → `03_Comparacao_Modelos.ipynb`
- Otimização bayesiana de hiperparâmetros com Optuna → `04_Otimizacao_Optuna.ipynb`